In [3]:
# ============================================
# 1. Install Required Library
# ============================================
!pip install imbalanced-learn

# ============================================
# 2. Import Libraries
# ============================================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import IsolationForest
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import classification_report, roc_auc_score

from imblearn.over_sampling import SMOTE


# ============================================
# 3. Upload and Load Dataset
# ============================================
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("creditcard_fraud_dataset.csv")

print("Dataset Shape:", df.shape)
print(df.head())


# ============================================
# 4. Separate Features and Target
# ============================================
X = df.drop("Class", axis=1)
y = df["Class"]


# ============================================
# 5. Train Test Split
# ============================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ============================================
# 6. Feature Scaling
# ============================================
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# ============================================
# 7. Handle Class Imbalance using SMOTE
# ============================================
smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Before SMOTE:")
print(y_train.value_counts())

print("After SMOTE:")
print(pd.Series(y_train_resampled).value_counts())


# ============================================
# 8. Unsupervised Anomaly Detection
# Isolation Forest
# ============================================
iso = IsolationForest(
    contamination=0.002,
    random_state=42
)

iso.fit(X_train)

train_anomaly_score = iso.decision_function(X_train_resampled) # Corrected to use X_train_resampled
test_anomaly_score = iso.decision_function(X_test)


# ============================================
# 9. Add Anomaly Score as Feature
# ============================================
train_score_scaled = np.interp(
    train_anomaly_score,
    (train_anomaly_score.min(), train_anomaly_score.max()),
    (0,1)
)

test_score_scaled = np.interp(
    test_anomaly_score,
    (test_anomaly_score.min(), test_anomaly_score.max()),
    (0,1)
)

X_train_hybrid = np.column_stack((X_train_resampled, train_score_scaled))
X_test_hybrid = np.column_stack((X_test, test_score_scaled))


# ============================================
# 10. Random Forest (Hybrid Model)
# ============================================
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train_hybrid, y_train_resampled)

rf_pred = rf.predict(X_test_hybrid)

print("\nRandom Forest Hybrid Model")
print(classification_report(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_pred))


# ============================================
# 11. Logistic Regression
# ============================================
lr = LogisticRegression(max_iter=1000)

lr.fit(X_train_hybrid, y_train_resampled)

lr_pred = lr.predict(X_test_hybrid)

print("\nLogistic Regression Hybrid Model")
print(classification_report(y_test, lr_pred))
print("ROC-AUC:", roc_auc_score(y_test, lr_pred))


# ============================================
# 12. Gradient Boosting
# ============================================
gb = GradientBoostingClassifier()

gb.fit(X_train_hybrid, y_train_resampled)

gb_pred = gb.predict(X_test_hybrid)

print("\nGradient Boosting Hybrid Model")
print(classification_report(y_test, gb_pred))
print("ROC-AUC:", roc_auc_score(y_test, gb_pred))


# ============================================
# 13. Pure Supervised Model (Comparison)
# ============================================
rf_pure = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_pure.fit(X_train_resampled, y_train_resampled)

pure_pred = rf_pure.predict(X_test)

print("\nPure Supervised Random Forest")
print(classification_report(y_test, pure_pred))
print("ROC-AUC:", roc_auc_score(y_test, pure_pred))


# ============================================
# 14. Final Comparison
# ============================================
print("\nConclusion:")
print("Hybrid model combines anomaly detection with supervised learning.")
print("This helps detect rare fraud patterns better than pure supervised models.")

Saving creditcard_fraud_dataset.csv to creditcard_fraud_dataset (2).csv
Dataset Shape: (9492, 31)
       Time        V1        V2        V3        V4        V5        V6  \
0 -1.347912 -0.168217  0.272817  1.086769  0.892330 -0.440059 -0.130394   
1  0.046680 -1.388845  1.597065 -0.049984 -1.733669  1.302984 -1.157159   
2  0.202484  0.870456  0.283521  0.926952 -0.781412 -0.109818 -0.694801   
3  0.789135 -0.910628 -1.350674 -0.372718  1.120916 -1.588297  0.084268   
4 -1.115598 -1.672509 -0.899634  0.864727 -0.499471  0.415164  0.417314   

         V7        V8        V9  ...       V21       V22       V23       V24  \
0  0.904764  0.939469 -0.088050  ... -0.570699 -0.490884  0.481454  1.568147   
1  0.255012  1.813793 -0.258105  ...  0.594375  0.810028  0.275471  0.452543   
2 -0.107931 -0.933780 -0.714796  ... -0.089465  2.206643  1.693560  0.334930   
3  0.441018  1.587563  0.678657  ... -0.878147  0.893470  1.017822 -2.145325   
4  0.889570 -0.646592 -2.551829  ... -0.478734  1.5